In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from pathlib import Path

ROOT = Path(
    '/content/drive/.shortcut-targets-by-id/'
    '1bvmgroOvYdtVKlMVWZ2iwf5ZGwogZ4xm/boneage_colab'
)

FRIEND_REPO = ROOT / 'friend_repo'
DATA = ROOT / 'data_goc'
SCRIPTS = ROOT / 'baseline_scripts'

print('ROOT:', ROOT)
print('friend_repo:', FRIEND_REPO.exists())
print('data_goc:', DATA.exists())
print('baseline_scripts:', SCRIPTS.exists())

ROOT: /content/drive/.shortcut-targets-by-id/1bvmgroOvYdtVKlMVWZ2iwf5ZGwogZ4xm/boneage_colab
friend_repo: True
data_goc: True
baseline_scripts: True


In [3]:
!nvidia-smi

Sun Aug 23 08:36:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

CUDA available: True
GPU: Tesla T4


In [11]:
import shutil

shutil.copytree(
    FRIEND_REPO,
    '/content/friend_repo',
    dirs_exist_ok=True
)

shutil.copytree(
    SCRIPTS,
    '/content/project/baseline_v1/scripts',
    dirs_exist_ok=True
)

'/content/project/baseline_v1/scripts'

In [12]:
%cd /content/friend_repo
!pip install -q -r p1_baseline/requirements.txt

/content/friend_repo


In [14]:
!python /content/project/baseline_v1/scripts/prepare_exp006_p7_5fold.py \
  --repo-root /content/friend_repo \
  --data-root "{DATA}" \
  --output-dir /content/exp006_roadmap \
  --run-output-root /content/exp006_roadmap/runs

{
  "created_at_utc": "2026-08-23T08:52:17.117003+00:00",
  "experiment": "EXP-006/007/008 five-fold roadmap preparation",
  "test_labels_used": false,
  "source_manifest": "/content/friend_repo/p0_audit/outputs/development_manifest_14036.csv",
  "source_manifest_sha256": "ee8f6ccd4b94b33ee21a11a0c5b6223f4f7adcb28c5cd3fcd3228f0c647776d5",
  "local_data_root": "/content/drive/.shortcut-targets-by-id/1bvmgroOvYdtVKlMVWZ2iwf5ZGwogZ4xm/boneage_colab/data_goc",
  "development_count": 14036,
  "folds": 5,
  "seed": 42,
  "stratification": "sex + age bins [0,60,120,180,229]",
  "run_output_root": "/content/exp006_roadmap/runs",
  "folds_detail": [
    {
      "fold": 1,
      "train_count": 11228,
      "val_count": 2808,
      "train_manifest": "/content/exp006_roadmap/fold_1/train_manifest.csv",
      "val_manifest": "/content/exp006_roadmap/fold_1/validation_manifest.csv",
      "train_hash": "f6c953ba9a0920657d7eee09311f4f325bd708317b026c333cecd03e83f58bf8",
      "val_hash": "61fdd2e96f4

In [16]:
%cd /content/friend_repo

!python -m p1_baseline.preflight \
  --config /content/exp006_roadmap/fold_1/p7_control.toml \
  --no-pretrained

/content/friend_repo
{
  "status": "PASS",
  "checks": {
    "train_count": true,
    "val_count": true,
    "train_hash": true,
    "val_hash": true,
    "test_path_absent": true,
    "sample_shape": true,
    "forward_shape": true
  },
  "config_hash": "edaa5c74c2b6126a81f816188a168b1d570e31f01444c0e020f39464b934cff0",
  "sample_id": "1377",
  "sample_target_months": 180.0
}


In [7]:
!cd /content/friend_repo && python -m p1_baseline.preflight \
  --config /content/drive/MyDrive/boneage_colab/exp004_friend_holdout/friend_p7_fresh_holdout.toml \
  --no-pretrained

{
  "status": "PASS",
  "checks": {
    "train_count": true,
    "val_count": true,
    "train_hash": true,
    "val_hash": true,
    "test_path_absent": true,
    "sample_shape": true,
    "forward_shape": true
  },
  "config_hash": "ff4d0d202b58f5e6065e330039accc6e0868a2ec6b076802642232c285d6c945",
  "sample_id": "10000",
  "sample_target_months": 96.0
}


In [8]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

CUDA available: True
GPU: Tesla T4


In [17]:
!ls /content/exp006_roadmap/fold_1

d3_ldl_fused.toml    p7_control.toml	 validation_manifest.csv
d3_ldl_regonly.toml  train_manifest.csv


In [24]:
!grep -n "checkpoint_mirror_root\|output_root\|run_id" \
  /content/exp006_roadmap/fold_1/p7_control.toml

9:run_id = "EXP006_P7_CONTROL_FOLD_1"
10:output_root = "/content/exp006_roadmap/runs"


In [25]:
from pathlib import Path
import json

cfg_path = Path('/content/exp006_roadmap/fold_1/p7_control.toml')
mirror = ROOT / 'outputs' / 'exp006_roadmap'

text = cfg_path.read_text()

if 'checkpoint_mirror_root' not in text:
    text = text.replace(
        '[run]\n',
        '[run]\n'
        f'checkpoint_mirror_root = {json.dumps(str(mirror))}\n',
        1
    )
    cfg_path.write_text(text)

print('Mirror:', mirror)
print('Đã cấu hình:', 'checkpoint_mirror_root' in cfg_path.read_text())

Mirror: /content/drive/.shortcut-targets-by-id/1bvmgroOvYdtVKlMVWZ2iwf5ZGwogZ4xm/boneage_colab/outputs/exp006_roadmap
Đã cấu hình: True


In [26]:
!grep -n "checkpoint_mirror_root" \
  /content/exp006_roadmap/fold_1/p7_control.toml

9:checkpoint_mirror_root = "/content/drive/.shortcut-targets-by-id/1bvmgroOvYdtVKlMVWZ2iwf5ZGwogZ4xm/boneage_colab/outputs/exp006_roadmap"


In [27]:
from pathlib import Path
import shutil

ROOT = Path(
    '/content/drive/.shortcut-targets-by-id/'
    '1bvmgroOvYdtVKlMVWZ2iwf5ZGwogZ4xm/boneage_colab'
)

run_dir = Path(
    '/content/exp006_roadmap/runs/'
    'EXP006_P7_CONTROL_FOLD_1'
)

backup_dir = (
    ROOT / 'outputs' / 'exp006_roadmap'
    / 'EXP006_P7_CONTROL_FOLD_1'
)

print('Run tồn tại:', run_dir.exists())
print('Checkpoint:', list(run_dir.glob('*.ckpt')))

if not run_dir.exists():
    raise FileNotFoundError(f'Không tìm thấy: {run_dir}')

shutil.copytree(run_dir, backup_dir, dirs_exist_ok=True)

print('Đã sao lưu lên Drive:', backup_dir)

Run tồn tại: True
Checkpoint: [PosixPath('/content/exp006_roadmap/runs/EXP006_P7_CONTROL_FOLD_1/last.ckpt'), PosixPath('/content/exp006_roadmap/runs/EXP006_P7_CONTROL_FOLD_1/best_mae.ckpt')]
Đã sao lưu lên Drive: /content/drive/.shortcut-targets-by-id/1bvmgroOvYdtVKlMVWZ2iwf5ZGwogZ4xm/boneage_colab/outputs/exp006_roadmap/EXP006_P7_CONTROL_FOLD_1


In [ ]:
%cd /content/friend_repo

!python -m p1_baseline.train \
  --config /content/exp006_roadmap/fold_1/p7_control.toml \
  --resume "{ROOT / 'outputs/exp006_roadmap/EXP006_P7_CONTROL_FOLD_1/last.ckpt'}"

/content/friend_repo
2026-08-23 11:08:04,272 | INFO | Run=EXP006_P7_CONTROL_FOLD_1 device=cuda config_hash=edaa5c74c2b6126a81f816188a168b1d570e31f01444c0e020f39464b934cff0
2026-08-23 11:08:04,273 | INFO | Split hash train=f6c953ba9a0920657d7eee09311f4f325bd708317b026c333cecd03e83f58bf8 validation=61fdd2e96f4f8c8f33e8728114604ad8ba5d1c4307ad22b4b4642786630368b9
RESUME CHECK
Run ID: EXP006_P7_CONTROL_FOLD_1
Checkpoint path: /content/drive/.shortcut-targets-by-id/1bvmgroOvYdtVKlMVWZ2iwf5ZGwogZ4xm/boneage_colab/outputs/exp006_roadmap/EXP006_P7_CONTROL_FOLD_1/last.ckpt
Epoch/global step tiếp tục: 5/1560
Best validation MAE trước đó: 6.965027377136752
Split hash khớp: CÓ
Config hash khớp: CÓ
Code version khớp: CÓ
Optimizer/scheduler/scaler đã phục hồi: CÓ
2026-08-23 11:08:37,778 | INFO | epoch=6 batch=25 global_step=1568 loss_reg_months=5.2223 lr=0.00019015 grad_norm=1.6862 throughput=9.16 img/s eta_seconds=1193.3 gpu_alloc=485.7MiB gpu_reserved=4426.0MiB skipped_batches=0
2026-08-23 11:08:5